# 05 — 消融实验与显著性检验

**负责人**: 石韫嘉 | **周次**: W15
**目标**: 量化各模态增量贡献，检验模型间差异显著性

## 1. 环境与数据加载

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import compute_metrics
from src.evaluation.significance_test import compare_absolute_errors, improvement_ratio

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11})
sns.set_style('whitegrid')

RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

df_results = pd.read_csv(RESULTS_DIR / 'experiment_log.csv')
print('Environment ready')

In [ ]:
# Load data and create consistent train/val/test split
df = pd.read_csv(DATA_PROCESSED / 'florida_structured.csv')
target_col = 'lastSoldPrice'
drop_cols = [target_col, 'listPrice']
for col in ['_id', '_split', 'sanitized_text', 'clean_text', 'type', 'sub_type', 'zip', 'address', 'description']:
    if col in df.columns:
        drop_cols.append(col)
feature_cols = [c for c in df.columns if c not in drop_cols]

X_struct = np.nan_to_num(df[feature_cols].values.astype(np.float64), nan=0.0, posinf=0.0, neginf=0.0)
y = df[target_col].values.astype(np.float64)
y = np.nan_to_num(y, nan=np.nanmedian(y))

tfidf = joblib.load(DATA_PROCESSED / 'florida_tfidf_features.pkl').astype(np.float64)
bert_emb = joblib.load(DATA_PROCESSED / 'florida_bert_embeddings.pkl').astype(np.float64)

indices = np.arange(len(y))
idx_train, idx_temp = train_test_split(indices, test_size=0.2, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42)

X_test = X_struct[idx_test]
y_test = y[idx_test]
X_text_test = {'tfidf': tfidf[idx_test].astype(np.float64), 'bert_embeddings': bert_emb[idx_test].astype(np.float64)}
print(f'Samples: Train={len(idx_train)}, Val={len(idx_val)}, Test={len(idx_test)}')

In [ ]:
# Load all trained models and generate test-set predictions
from src.models.structured_baseline import LinearBaseline, RandomForestBaseline, XGBoostBaseline
from src.models.text_baseline import TFIDFRidgeBaseline, BERTMLPBaseline
from src.models.early_fusion import EarlyFusionXGBoost, EarlyFusionMLP
from src.models.mid_fusion import MidFusionModel
from src.models.late_fusion import LateFusionStacking
from sklearn.preprocessing import StandardScaler

# Get training targets for repairing PyTorch model scalers
y_train_arr = y[idx_train]

# Mapping: display_name -> (class, modality, model_filename)
all_model_specs = [
    ('LinearReg', LinearBaseline, 'structured', 'LinearBaseline.joblib'),
    ('RandomForest', RandomForestBaseline, 'structured', 'RandomForestBaseline.joblib'),
    ('XGBoost', XGBoostBaseline, 'structured', 'XGBoostBaseline.joblib'),
    ('TFIDF+Ridge', TFIDFRidgeBaseline, 'text', 'TFIDFRidgeBaseline.joblib'),
    ('BERT+MLP', BERTMLPBaseline, 'text', 'BERTMLPBaseline.joblib'),
    ('EarlyFusionXGBoost', EarlyFusionXGBoost, 'fusion_early', 'EarlyFusionXGBoost.joblib'),
    ('EarlyFusionMLP', EarlyFusionMLP, 'fusion_early', 'EarlyFusionMLP.joblib'),
    ('MidFusion', MidFusionModel, 'fusion_mid', 'MidFusionModel.joblib'),
    # LateFusionStacking has save/load issues; skip loading, use CSV metrics
]

predictions = {}
for name, cls, modality, filename in all_model_specs:
    model_path = MODELS_DIR / filename
    if not model_path.exists():
        print(f'Skip {name}: file {filename} not found')
        continue
    try:
        model = cls.load(str(model_path))
        # Repair _y_scaler for PyTorch models (lost during serialization)
        if hasattr(model, '_y_scaler') and model._y_scaler is None:
            model._y_scaler = StandardScaler()
            model._y_scaler.fit(y_train_arr.reshape(-1, 1))
        if modality == 'structured':
            pred = model.predict(X_test)
        elif modality == 'text':
            pred = model.predict(None, X_text=X_text_test)
        else:
            pred = model.predict(X_test, X_text=X_text_test)
        predictions[name] = pred
        m = compute_metrics(y_test, pred)
        print(f'{name} ({modality}): RMSE=${m["rmse"]:,.0f}  R²={m["r2"]:.4f}')
    except Exception as e:
        print(f'{name}: error - {e}')

print(f'\nLoaded {len(predictions)} models for analysis')

## 2. 消融对比：仅结构化 vs 仅文本 vs 融合

In [ ]:
# Modality-level performance summary (from CSV for consistency with LateFusionStacking)
modality_map = {
    'structured': 'Structured Only', 'text': 'Text Only',
    'fusion_early': 'Early Fusion', 'fusion_mid': 'Mid Fusion', 'fusion_late': 'Late Fusion',
}
ablation_order = ['Structured Only', 'Text Only', 'Early Fusion', 'Mid Fusion', 'Late Fusion']

# Build ablation data using CSV results (all models, even those that can't load)
ablation_data = []
for _, row in df_results.iterrows():
    if row['status'] != 'success':
        continue
    modality_label = modality_map.get(row['modality'], row['modality'])
    short_name = row['model_name'].replace('Baseline', '')
    ablation_data.append({
        'Model': short_name, 'Modality': modality_label,
        'RMSE': row['test_rmse'], 'MAE': row['test_mae'],
        'R²': row['test_r2'], 'MAPE': row['test_mape'],
    })

df_ablation = pd.DataFrame(ablation_data)
df_ablation['Modality'] = pd.Categorical(df_ablation['Modality'], categories=ablation_order, ordered=True)
df_ablation.sort_values(['Modality', 'RMSE']).reset_index(drop=True)

In [ ]:
# ============================================================
# FIGURE 1: Ablation — RMSE/MAE/R²/MAPE by modality group
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
modality_colors_5 = ['#4472C4', '#ED7D31', '#70AD47', '#A855F7', '#E74C3C']

for (metric, label, fmt, ax) in [
    ('RMSE', 'RMSE ($)', '${:,.0f}', axes[0,0]),
    ('MAE', 'MAE ($)', '${:,.0f}', axes[0,1]),
    ('R²', 'R²', '{:.4f}', axes[1,0]),
    ('MAPE', 'MAPE (%)', '{:.1f}%', axes[1,1]),
]:
    modality_means = df_ablation.groupby('Modality')[metric].mean()
    modality_means = modality_means.reindex([o for o in ablation_order if o in modality_means.index])
    bars = ax.bar(range(len(modality_means)), modality_means.values,
                  color=[modality_colors_5[i] for i in range(len(modality_means))],
                  edgecolor='white', lw=1)
    ax.set_xticks(range(len(modality_means)))
    ax.set_xticklabels(modality_means.index, rotation=20, ha='right', fontsize=9)
    ax.set_title(label, fontweight='bold')
    for b, v in zip(bars, modality_means.values):
        if v > 0:
            offset = max(modality_means.values) * 0.02 if max(modality_means.values) > 0 else 100
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+offset,
                    fmt.format(v), ha='center', va='bottom', fontsize=10, fontweight='bold')

fig.suptitle('Ablation Study: Modality Contribution to Prediction Performance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_ablation_modality.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# FIGURE 2: Model-level RMSE heatmap
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))
heat_data = df_ablation.pivot(index='Model', columns='Modality', values='RMSE')
heat_data_filled = heat_data.fillna(0)
sns.heatmap(heat_data_filled, annot=True, fmt=',.0f', cmap='RdYlGn_r',
            linewidths=1, linecolor='white', cbar_kws={'label': 'RMSE ($)', 'shrink': 0.8},
            ax=ax, annot_kws={'fontsize': 9})
ax.set_title('RMSE Heatmap: Models x Modalities', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_ablation_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Quantify incremental contribution of each modality
print('=' * 70)
print('INCREMENTAL CONTRIBUTION QUANTIFICATION')
print('=' * 70)

best_struct_rmse = df_ablation[df_ablation['Modality']=='Structured Only']['RMSE'].min()
best_text_rmse = df_ablation[df_ablation['Modality']=='Text Only']['RMSE'].min()
best_early_rmse = df_ablation[df_ablation['Modality']=='Early Fusion']['RMSE'].min()
best_mid_rmse = df_ablation[df_ablation['Modality']=='Mid Fusion']['RMSE'].min()
best_late_rmse = df_ablation[df_ablation['Modality']=='Late Fusion']['RMSE'].min()

print(f'\nBest RMSE per modality:')
print(f'  Structured Only:  ${best_struct_rmse:,.0f}')
print(f'  Text Only:        ${best_text_rmse:,.0f}')
print(f'  Early Fusion:     ${best_early_rmse:,.0f}')
print(f'  Mid Fusion:       ${best_mid_rmse:,.0f}')
print(f'  Late Fusion:      ${best_late_rmse:,.0f}')

print(f'\nFusion gain over best single-modality (Structured):')
print(f'  Early Fusion:     {improvement_ratio(best_struct_rmse, best_early_rmse):.1f}% RMSE reduction')
print(f'  Mid Fusion:       {improvement_ratio(best_struct_rmse, best_mid_rmse):.1f}% RMSE reduction')
print(f'  Late Fusion:      {improvement_ratio(best_struct_rmse, best_late_rmse):.1f}% RMSE reduction')

print(f'\nFusion gain over best single-modality (Text):')
print(f'  Early Fusion:     {improvement_ratio(best_text_rmse, best_early_rmse):.1f}% RMSE reduction')
print(f'  Mid Fusion:       {improvement_ratio(best_text_rmse, best_mid_rmse):.1f}% RMSE reduction')
print(f'  Late Fusion:      {improvement_ratio(best_text_rmse, best_late_rmse):.1f}% RMSE reduction')

## 3. 性能瀑布图：单模态 → 融合

In [ ]:
# ============================================================
# FIGURE 3: Waterfall chart — RMSE from baseline to fusion
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

waterfall_labels = ['Best\nStructured', 'Best\nText', 'Early\nFusion', 'Mid\nFusion', 'Late\nFusion']
waterfall_rmse = [best_struct_rmse, best_text_rmse, best_early_rmse, best_mid_rmse, best_late_rmse]
waterfall_colors = ['#4472C4', '#ED7D31', '#70AD47', '#A855F7', '#E74C3C']

bars = ax.bar(waterfall_labels, waterfall_rmse, color=waterfall_colors, edgecolor='white', lw=1.5)
for b, v in zip(bars, waterfall_rmse):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+2000,
            f'${v:,.0f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Improvement arrows from baseline
for i in range(2, len(waterfall_rmse)):
    if waterfall_rmse[i] < best_struct_rmse:
        pct = (best_struct_rmse - waterfall_rmse[i]) / best_struct_rmse * 100
        ax.annotate(f'↓{pct:.1f}%', xy=(i, waterfall_rmse[i]),
                    xytext=(i, waterfall_rmse[i] - 25000),
                    ha='center', fontsize=12, color='darkgreen', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='darkgreen', lw=2))

ax.set_ylabel('RMSE ($)')
ax.set_title('Model Performance Waterfall: Single-Modality → Fusion', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_ablation_waterfall.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. 过拟合诊断：训练 vs 测试 RMSE

In [ ]:
# ============================================================
# FIGURE 4: Train vs Test RMSE scatter (overfitting diagnostic)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

success_df = df_results[df_results['status'] == 'success'].copy()
color_map = {'structured': '#4472C4', 'text': '#ED7D31',
             'fusion_early': '#70AD47', 'fusion_mid': '#A855F7', 'fusion_late': '#E74C3C'}

for _, row in success_df.iterrows():
    color = color_map.get(row['modality'], 'grey')
    ax.scatter(row['train_rmse'], row['test_rmse'], s=150, c=color, edgecolors='white', lw=1.5,
               zorder=5, alpha=0.85)
    short_name = row['model_name'].replace('Baseline','')
    ax.annotate(short_name, (row['train_rmse'], row['test_rmse']),
                textcoords='offset points', xytext=(8, 5), fontsize=7.5, alpha=0.9)

all_vals = np.concatenate([success_df['train_rmse'].values, success_df['test_rmse'].values])
lim_low, lim_high = np.min(all_vals) * 0.7, np.max(all_vals) * 1.1
ax.plot([lim_low, lim_high], [lim_low, lim_high], 'k--', alpha=0.3, lw=1, label='No overfitting')
ax.set_xlim(lim_low, lim_high); ax.set_ylim(lim_low, lim_high)
ax.set_xlabel('Train RMSE ($)'); ax.set_ylabel('Test RMSE ($)')
ax.set_title('Train vs Test RMSE — Overfitting Diagnostic', fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
leg = [Patch(facecolor=c, label=l.replace('fusion_','').title().replace('_',' '))
       for l, c in color_map.items()]
ax.legend(handles=leg, fontsize=8, loc='lower right')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_ablation_learning_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print('Points above diagonal → overfitting (test RMSE > train RMSE)')
print('Points below diagonal → good generalization')

## 5. 显著性检验：Wilcoxon 与 Paired t-test

In [ ]:
# Pairwise significance between key model pairs
key_pairs = [
    ('XGBoost', 'TFIDF+Ridge'),
    ('XGBoost', 'EarlyFusionMLP'),
    ('TFIDF+Ridge', 'EarlyFusionMLP'),
    ('XGBoost', 'MidFusion'),
    ('EarlyFusionMLP', 'MidFusion'),
    ('MidFusion', 'LateFusionStacking'),
    ('EarlyFusionXGBoost', 'EarlyFusionMLP'),
]

results = []
for model_a, model_b in key_pairs:
    if model_a not in predictions or model_b not in predictions:
        continue
    r_w = compare_absolute_errors(y_test, predictions[model_a], predictions[model_b], method='wilcoxon')
    r_t = compare_absolute_errors(y_test, predictions[model_a], predictions[model_b], method='ttest')
    for method_name, r in [('Wilcoxon', r_w), ('Paired t-test', r_t)]:
        results.append({
            'Model A': model_a, 'Model B': model_b,
            'Method': method_name,
            'Statistic': round(r.statistic, 2),
            'p-value': f'{r.p_value:.2e}',
            'α': 0.05,
            'Significant': 'YES (p<0.05)' if r.significant else 'Not significant',
        })

df_sig = pd.DataFrame(results)
df_sig

In [ ]:
# ============================================================
# FIGURE 5: Pairwise significance matrix (Wilcoxon p-values)
# ============================================================
model_names_ordered = ['XGBoost', 'TFIDF+Ridge', 'EarlyFusionXGBoost',
                        'EarlyFusionMLP', 'MidFusion', 'LateFusionStacking']
available_models = [m for m in model_names_ordered if m in predictions]
n = len(available_models)
p_matrix = np.ones((n, n))

for i, ma in enumerate(available_models):
    for j, mb in enumerate(available_models):
        if ma in predictions and mb in predictions:
            r = compare_absolute_errors(y_test, predictions[ma], predictions[mb], method='wilcoxon')
            p_matrix[i, j] = r.p_value

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(p_matrix, dtype=bool), k=1)
annot_matrix = np.empty_like(p_matrix, dtype=object)
for i in range(n):
    for j in range(n):
        if mask[i, j]:
            annot_matrix[i, j] = ''
        elif p_matrix[i, j] < 0.001:
            annot_matrix[i, j] = '<0.001'
        elif p_matrix[i, j] < 0.01:
            annot_matrix[i, j] = f'{p_matrix[i,j]:.4f}'
        elif p_matrix[i, j] < 0.05:
            annot_matrix[i, j] = f'{p_matrix[i,j]:.4f}'
        else:
            annot_matrix[i, j] = f'{p_matrix[i,j]:.3f}'

sns.heatmap(p_matrix, annot=annot_matrix, fmt='', cmap='RdYlGn_r', vmin=0, vmax=1,
            xticklabels=available_models, yticklabels=available_models,
            mask=mask, linewidths=1, linecolor='white',
            cbar_kws={'label': 'Wilcoxon p-value', 'shrink': 0.8}, ax=ax)
ax.set_title('Pairwise Significance Matrix (Wilcoxon Signed-Rank Test)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'fig_ablation_significance_matrix.png', dpi=200, bbox_inches='tight')
plt.show()
print('Green cells = p < 0.05 (statistically significant difference)')

## 6. 消融实验总结

### 各模态增量贡献量化

| Ablation Condition | Best Model | RMSE | R² | vs Best Structured |
|-------------------|-----------|------|-----|-------------------|
| Structured Only | XGBoost | $142,935 | 0.727 | baseline |
| Text Only | TF-IDF+Ridge | $175,621 | 0.587 | -22.9% worse |
| Early Fusion | EarlyFusionMLP | $107,725 | 0.845 | +24.6% better |
| Mid Fusion | MidFusion | $118,124 | 0.813 | +17.4% better |
| Late Fusion | LateFusionStacking | $137,795 | 0.746 | +3.6% better |

### 核心结论

1. **Early Fusion (EarlyFusionMLP) 表现最佳**: RMSE 比最佳单模态降低 24.6%，文本提供显著增量
2. **Mid Fusion (Attention) 次之**: 注意力融合优于晚期 Stacking
3. **Late Fusion 改进有限**: Stacking 仅比 XGBoost 单模态提升 3.6%
4. **融合模型均优于任何单模态模型**: 证明了多源数据融合的有效性
5. **MLP 融合优于 XGBoost 融合**: 说明深度网络能更好地捕获跨模态非线性交互